# Dynamic Semantics with Continuations

This workbook exemplifies the idea of handling dynamic semantics using continuations, as is done e.g. in the following papers:

- de Groote, Philippe (2006). [Towards a Montagovian Account of Dynamics](https://journals.linguisticsociety.org/proceedings/index.php/SALT/article/view/2952/2692). In Masayuki Gibson & Jonathan Howell (eds.), _Proceedings of SALT_ 16:1–16.
- Groenendijk, Jeroen & Martin Stokhof (1990). [Dynamic Montague Grammar](https://eprints.illc.uva.nl/id/eprint/1148/). Technical Report LP-1990-02, University of Amsterdam.

The underlying idea in these papers is that if $a$ is your type for tracking discourse referents, then sentence meanings are of type $\langle a,\langle\langle a,t\rangle,t\rangle$. That means that in principle, a sentence $S$ can place the sentence following it within the scope of operators contained in $S$, which is how you get dynamic binding.

In the implementation below, $a$ is polymorphic. To achieve the effect of tracking discourse referents by means of lists, without a native list type, I mimic the effect of appending object $o$ to dicourse &lsquo;list&rsquo; $d$ by forming the pair $(o,d)$; it follows that this action changes the type of the discourse.

In [1]:
composition_system = lang.hk_system.copy()
display(meta.get_type_system())
lang.get_system()

Type system with atomic types: $e, t, n$

Composition system 'Type-driven composition'
Operations: {FA, PM, PA, VAC}

## Lexicon

In [2]:
%%lamb reset, ambiguity
# determiners
||a|| = L n_<e,<X,<<Y,t>,t>>>: L v_<e,<(e,Y),<<Z,t>,t>>>: L i_X: L f_<Z,t>: Exists x_e: n(x)(i)(L j_Y: v(x)((x,j))(f))
||a|| = L n_<e,<X3,<<Y3,t>,t>>>: L v_<e,<(e,Y3),<<Z3,t>,t>>>: L i_X3: L f_<Z,t>: Exists x_e: n(x)(i)(L j_Y3: v(x)((x,j))(f))
||every|| = L n_<e,<X,<<Y3,t>,t>>>: L v_<e,<(e,Y3),<<Z,t>,t>>>: L i_X: L f_<X,t>: ~(Exists x_e: n(x)(i)(L j_Y3: ~v(x)((x,j))(L k_Z: True))) & f(i)
# nouns
||farmer|| = L x_e: L i_Y: L f_<Y,t>: Farmer_<e,t>(x) & f(i)
||donkey|| = L x_e: L i_Y: L f_<Y,t>: Donkey_<e,t>(x) & f(i)
# verbs
||brayed|| = L x_e: L i_Y: L f_<Y,t>: Bray_<e,t>(x) & f(i)
||snorted|| = L x_e: L i_Y: L f_<Y,t>: Snort_<e,t>(x) & f(i)
||owns|| = L y_e: L x_e: L i_Y: L f_<Y,t>: Own_<(e,e),t>(x,y) & f(i)
||feeds|| = L y_e: L x_e: L i_Y: L f_<Y,t>: Feed_<(e,e),t>(x,y) & f(i)
# connectives
||aND|| = L q_<X,<<Z,t>,t>>: L p_<Y,<<X,t>,t>>: L i_Y: L f_<Z,t>: p(i)(L j_X: q(j)(f))
||iF|| = L p_<Y2,<<X2,t>,t>>: L q_<X2,<<Z2,t>,t>>: L i_Y2: L f_<Y2,t>: ~p(i)(L j_X2: ~q(j)(L k_Z2: True)) & f(i)
# ||who|| = L v_<e,<(e,X),<<Z,t>,t>>>: L n_<e,<X,<<X,t>,t>>>: L x_e: L i_X: L f_<Z,t>: n(x)(i)(L j_X: v(x)((x,j))(f)) see note below
||who|| = L v_<e,<X5,<<Z5,t>,t>>>: L n_<e,<X5,<<X5,t>,t>>>: L x_e: L i_X5: L f_<Z,t>: n(x)(i)(L j_X5: v(x)(j)(f))
# pronouns
||he|| = L v_<e,<(e,X1),<<Y,t>,t>>>: L i_(e,X1): v(i[0])(i) # pron0
||it|| = L v_<e,<(e,X1),<<Y,t>,t>>>: L i_(e,X1): v(i[0])(i) # pron0
||he|| = L v_<e,<(Y1,(e,X4)),<<Y,t>,t>>>: L i_(Y1,(e,X1)): v(i[1][0])(i) # pron1
||it|| = L v_<e,<(Y1,(e,X4)),<<Y,t>,t>>>: L i_(Y1,(e,X1)): v(i[1][0])(i) # pron1
# closure
||E|| = L p_<{e},<<Y6,t>,t>>: p({})(L c_Y6: True)
||C|| = L p_t: p

$[\![\text{\textbf{a[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, Y\right),\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, Y\right),\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}Z,t\right\rangle{}} \: . \: \exists{} x_{e} \: . \: {n}({x})({i})(\lambda{} j_{Y} \: . \: {v}({x})({x}, {j})({f}))$
<br />$[\![\text{\textbf{a[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X''',\left\langle{}\left\langle{}Y''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, Y'''\right),\left\langle{}\left\langle{}Z''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left\langle{}Z''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X''',\left\langle{}\left\langle{}Y''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, Y'''\right),\left\langle{}\left\langle{}Z''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} f_{\left\langle{}Z''',t\right\rangle{}} \: . \: \exists{} x_{e} \: . \: {n}({x})({i})(\lambda{} j_{Y'''} \: . \: {v}({x})({x}, {j})({f}))$<br />
$[\![\text{\textbf{every}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}Y''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, Y'''\right),\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}Y''',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, Y'''\right),\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}X,t\right\rangle{}} \: . \: {f}({i}) \wedge{} \neg{} (\exists{} x_{e} \: . \: {n}({x})({i})(\lambda{} j_{Y'''} \: . \: \neg{} {v}({x})({x}, {j})(\lambda{} k_{Z} \: . \: \textsf{True})))$<br />
$[\![\text{\textbf{farmer}}]\!]^{}_{\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Farmer}({x}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{donkey}}]\!]^{}_{\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Donkey}({x}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{brayed}}]\!]^{}_{\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Bray}({x}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{snorted}}]\!]^{}_{\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Snort}({x}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{owns}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Own}({x}, {y}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{feeds}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}Y,\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Y,t\right\rangle{}} \: . \: {Feed}({x}, {y}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{aND}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}Y,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}Y,\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} q_{\left\langle{}X,\left\langle{}\left\langle{}Z,t\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} p_{\left\langle{}Y,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{Y} \: . \: \lambda{} f_{\left\langle{}Z,t\right\rangle{}} \: . \: {p}({i})(\lambda{} j_{X} \: . \: {q}({j})({f}))$<br />
$[\![\text{\textbf{iF}}]\!]^{}_{\left\langle{}\left\langle{}Y'',\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X'',\left\langle{}\left\langle{}Z'',t\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}Y'',\left\langle{}\left\langle{}Y'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} p_{\left\langle{}Y'',\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} q_{\left\langle{}X'',\left\langle{}\left\langle{}Z'',t\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{Y''} \: . \: \lambda{} f_{\left\langle{}Y'',t\right\rangle{}} \: . \: {f}({i}) \wedge{} \neg{} {p}({i})(\lambda{} j_{X''} \: . \: \neg{} {q}({j})(\lambda{} k_{Z''} \: . \: \textsf{True}))$<br />
$[\![\text{\textbf{who}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X_{5},\left\langle{}\left\langle{}Z_{5},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}X_{5},\left\langle{}\left\langle{}X_{5},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}e,\left\langle{}X_{5},\left\langle{}\left\langle{}Z_{5},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}X_{5},\left\langle{}\left\langle{}Z_{5},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}e,\left\langle{}X_{5},\left\langle{}\left\langle{}X_{5},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X_{5}} \: . \: \lambda{} f_{\left\langle{}Z_{5},t\right\rangle{}} \: . \: {n}({x})({i})(\lambda{} j_{X_{5}} \: . \: {v}({x})({j})({f}))$<br />
$[\![\text{\textbf{he[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(e, X'\right)} \: . \: {v}({i}[\textsf{0}])({i})$
<br />$[\![\text{\textbf{he[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(Y', \left(e, X_{4}\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{0}])({i})$<br />
$[\![\text{\textbf{it[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(e, X'\right)} \: . \: {v}({i}[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(Y', \left(e, X_{4}\right)\right),\left\langle{}\left\langle{}Y,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(Y', \left(e, X_{4}\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{0}])({i})$<br />
$[\![\text{\textbf{E}}]\!]^{}_{\left\langle{}\left\langle{}\left\{e\right\},\left\langle{}\left\langle{}Y_{6},t\right\rangle{},t\right\rangle{}\right\rangle{},t\right\rangle{}} \:=\: \lambda{} p_{\left\langle{}\left\{e\right\},\left\langle{}\left\langle{}Y_{6},t\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: {p}(\{\}_{\left\{e\right\}})(\lambda{} c_{Y_{6}} \: . \: \textsf{True})$<br />
$[\![\text{\textbf{C}}]\!]^{}_{\left\langle{}t,t\right\rangle{}} \:=\: \lambda{} p_{t} \: . \: {p}$

N.B. I've followed de Groote in not passing the discourse referent for the noun into the relative clause that modifies it. I'm not sure that this is correct &mdash; you might want to say something like &lsquo;a man who owns a cat that scratched him&rsquo; &mdash; but it's simpler for current purposes.

In [3]:
# Traces and binders
t = [lang.Trace(n) for n in range(5)]
b = [lang.Binder(n) for n in range(5)]

In [4]:
counter = 0
def eg(example:str, counter:int) -> None:
    print(f'({counter})\t{example}\n')

def exemplify(example:str, counter:int, compResult) -> None:
    eg(example, counter)
    if len(compResult)<1:
        raise Exception("Composition failed for some reason.")
    print(compResult.source)
    print("")
    if len(compResult)<2:
        display(compResult.content[0].content)
    else:
        for n,cont in enumerate(compResult.content):
            print(f"Result [{n}]:")
            display(cont.content)

## Basic Examples

In [5]:
counter += 1
exemplify("A donkey brayed.", counter, (C * (E * ((a * donkey) * brayed))))

(1)	A donkey brayed.

[C [E [[a donkey] brayed]]]



(Exists x_e: (Bray_<e,t>(x_e) & Donkey_<e,t>(x_e)))

In [6]:
counter += 1
exemplify("Every donkey brayed.", counter, (C * (E * ((every * donkey) * brayed))))

(2)	Every donkey brayed.

[C [E [[every donkey] brayed]]]



¬(Exists x_e: (~Bray_<e,t>(x_e) & Donkey_<e,t>(x_e)))

In [7]:
counter += 1
exemplify("A donkey brayed aND it snorted.", counter, (C * (E * (((a * donkey) * brayed) * (aND * (it * snorted))))))

(3)	A donkey brayed aND it snorted.

[C [E [[[a donkey] brayed] [aND [it snorted]]]]]



(Exists x_e: ((Bray_<e,t>(x_e) & Donkey_<e,t>(x_e)) & Snort_<e,t>(x_e)))

In [8]:
counter += 1
exemplify("If a donkey brayed, it snorted.", counter, (C * (E * ((iF * ((a * donkey) * brayed)) * (it * snorted)))))

(4)	If a donkey brayed, it snorted.

[C [E [[iF [[a donkey] brayed]] [it snorted]]]]



¬(Exists x_e: ((Bray_<e,t>(x_e) & Donkey_<e,t>(x_e)) & ~Snort_<e,t>(x_e)))

In [9]:
(C * (E * ((iF * ((a * donkey) * brayed)) * (it * snorted))))

1 composition path.  Result:<br />
&nbsp;&nbsp;&nbsp;&nbsp;[0]: $[\![\text{\textbf{[C [E [[iF [[a donkey] brayed]] [it snorted]]]]}}]\!]^{}_{t} \:=\: \neg{} (\exists{} x_{e} \: . \: {Bray}({x}) \wedge{} {Donkey}({x}) \wedge{} \neg{} {Snort}({x}))$

In [10]:
counter += 1
exemplify("A farmer owns a donkey.", counter, (C * (E * ((a * farmer) * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2])))))))))

(5)	A farmer owns a donkey.

[C [E [[a farmer] [1 [[a donkey] [2 [t1 [owns t2]]]]]]]]



(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))))

In [11]:
counter += 1
exemplify("A farmer owns a donkey. He feeds it.", counter, (C * (E * (((a * farmer) * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2])))))) * (aND * (he * (b[3] * (it * (b[4] * (t[3] * (feeds * t[4])))))))))))
print("\nResult [0] is the intended interpretation.\n")

(6)	A farmer owns a donkey. He feeds it.

[C [E [[[a farmer] [1 [[a donkey] [2 [t1 [owns t2]]]]]] [aND [he [3 [it [4 [t3 [feeds t4]]]]]]]]]]

Result [0]:


(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x2_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [1]:


(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [2]:


(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))))


Result [0] is the intended interpretation.



In [12]:
counter += 1
exemplify("If a farmer owns a donkey, he feeds it.", counter, (C * (E * ((iF * ((a * farmer) * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2]))))))) * (he * (b[3] * (it * (b[4] * (t[3] * (feeds * t[4]))))))))))
print("\nResult [2] is the intended interpretation.\n")

(7)	If a farmer owns a donkey, he feeds it.

[C [E [[iF [[a farmer] [1 [[a donkey] [2 [t1 [owns t2]]]]]]] [he [3 [it [4 [t3 [feeds t4]]]]]]]]]

Result [0]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x2_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [1]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x2_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [2]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [3]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))))


Result [2] is the intended interpretation.



In [13]:
counter += 1
exemplify("Every farmer who owns a donkey feeds it.", counter, (C * (E * ((every * (farmer * (who * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2])))))))) * (b[3] * (it * (b[4] * (t[3] * (feeds * t[4])))))))))
print("\nResult [1] is the intended interpretation.\n")

(8)	Every farmer who owns a donkey feeds it.

[C [E [[every [farmer [who [1 [[a donkey] [2 [t1 [owns t2]]]]]]]] [3 [it [4 [t3 [feeds t4]]]]]]]]

Result [0]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))))

Result [1]:


¬(Exists x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))))


Result [1] is the intended interpretation.



## Strong and Weak Readings

The lexical entry for _every_ given above derives strong readings only (every farmer who owns a donkey feeds _every donkey thathe owns_), and we haven't had anything to say yet about other determiners.

In [14]:
%%lamb
detweak = L d_<({e},{e}),t>: L n_<e,<X4,<<Y4,t>,t>>>: L v_<e,<(e,Y4),<<Z4,t>,t>>>: L i_X4: L f_<X4,t>: d((Set x_e: n(x)(i)(L j_Y4: True)),(Set x_e: n(x)(i)(L j_Y4: v(x)((x,j))(L k_Z4: True)))) & f(i)
detstrong = L d_<({e},{e}),t>: L n_<e,<X4,<<Y4,t>,t>>>: L v_<e,<(e,Y4),<<Z4,t>,t>>>: L i_X4: L f_<X4,t>: d((Set x_e: n(x)(i)(L j_Y4: True)),(Set x_e: ~(n(x)(i)(L j_Y4: ~v(x)((x,j))(L k_Z4: True))))) & f(i)
||every|| = detweak(L u_({e},{e}): u[0] <= u[1])
||every[*]|| = detstrong(L u_({e},{e}): u[0] <= u[1])
||most|| = detweak(L u_({e},{e}): 2 * Card_<{e},n>(u[0] & u[1]) > Card_<{e},n>(u[0]))
||most[*]|| = detstrong(L u_({e},{e}): 2 * Card_<{e},n>(u[0] & u[1]) > Card_<{e},n>(u[0]))

${detweak}_{\left\langle{}\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}X_{4},\left\langle{}\left\langle{}Y_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, Y_{4}\right),\left\langle{}\left\langle{}Z_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left\langle{}X_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}}\:=\:\lambda{} d_{\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}e,\left\langle{}X_{4},\left\langle{}\left\langle{}Y_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, Y_{4}\right),\left\langle{}\left\langle{}Z_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} f_{\left\langle{}X_{4},t\right\rangle{}} \: . \: {d}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{Y_{4}} \: . \: \textsf{True})\}, \{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{Y_{4}} \: . \: {v}({x}_{e})({x}_{e}, {j})(\lambda{} k_{Z_{4}} \: . \: \textsf{True}))\}) \wedge{} {f}({i})$<br />
${detstrong}_{\left\langle{}\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}X_{4},\left\langle{}\left\langle{}Y_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, Y_{4}\right),\left\langle{}\left\langle{}Z_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left\langle{}X_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}}\:=\:\lambda{} d_{\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}e,\left\langle{}X_{4},\left\langle{}\left\langle{}Y_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, Y_{4}\right),\left\langle{}\left\langle{}Z_{4},t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} f_{\left\langle{}X_{4},t\right\rangle{}} \: . \: {d}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{Y_{4}} \: . \: \textsf{True})\}, \{{x}_{e} \:|\: \neg{} {n}({x}_{e})({i})(\lambda{} j_{Y_{4}} \: . \: \neg{} {v}({x}_{e})({x}_{e}, {j})(\lambda{} k_{Z_{4}} \: . \: \textsf{True}))\}) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{every[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}X,t\right\rangle{}} \: . \: (\forall{} x_{e} \: . \: {n}({x})({i})(\lambda{} j_{X'} \: . \: \textsf{True}) \rightarrow{} {n}({x})({i})(\lambda{} j_{X'} \: . \: {v}({x})({x}, {j})(\lambda{} k_{X''} \: . \: \textsf{True}))) \wedge{} {f}({i})$
<br />$[\![\text{\textbf{every[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}X,t\right\rangle{}} \: . \: (\forall{} x_{e} \: . \: {n}({x})({i})(\lambda{} j_{X'} \: . \: \textsf{True}) \rightarrow{} \neg{} {n}({x})({i})(\lambda{} j_{X'} \: . \: \neg{} {v}({x})({x}, {j})(\lambda{} k_{X''} \: . \: \textsf{True}))) \wedge{} {f}({i})$<br />
$[\![\text{\textbf{most[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}X,t\right\rangle{}} \: . \: ((\textsf{2} * {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: \textsf{True}) \wedge{} {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: {v}({x}_{e})({x}_{e}, {j})(\lambda{} k_{X''} \: . \: \textsf{True}))\})) > {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: \textsf{True})\})) \wedge{} {f}({i})$
<br />$[\![\text{\textbf{most[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}X,t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}e,\left\langle{}X,\left\langle{}\left\langle{}X',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X'\right),\left\langle{}\left\langle{}X'',t\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}X,t\right\rangle{}} \: . \: ((\textsf{2} * {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: \textsf{True}) \wedge{} \neg{} {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: \neg{} {v}({x}_{e})({x}_{e}, {j})(\lambda{} k_{X''} \: . \: \textsf{True}))\})) > {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({x}_{e})({i})(\lambda{} j_{X'} \: . \: \textsf{True})\})) \wedge{} {f}({i})$

In [15]:
exemplify("Every farmer who owns a donkey feeds it.", counter, (C * (E * ((every * (farmer * (who * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2])))))))) * (b[3] * (it * (b[4] * (t[3] * (feeds * t[4])))))))))
print("\nResult [1] is the weak reading we're after, and Result [3] is the strong interpretation (equivalent to the above).\n")

(8)	Every farmer who owns a donkey feeds it.

[C [E [[every [farmer [who [1 [[a donkey] [2 [t1 [owns t2]]]]]]]] [3 [it [4 [t3 [feeds t4]]]]]]]]

Result [0]:


(Forall x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))) >> (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e))))))

Result [1]:


(Forall x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))) >> (Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e))))))

Result [2]:


(Forall x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))) >> ~(Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e))))))

Result [3]:


(Forall x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))) >> ~(Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e))))))


Result [1] is the weak reading we're after, and Result [3] is the strong interpretation (equivalent to the above).



In [16]:
counter += 1
exemplify("Most farmers who own a donkey feeds it.", counter, (C * (E * ((most * (farmer * (who * (b[1] * ((a * donkey) * (b[2] * (t[1] * (owns * t[2])))))))) * (b[3] * (it * (b[4] * (t[3] * (feeds * t[4])))))))))
print("\nResult [1] is the weak reading we're after, and Result [3] is the strong interpretation.\n")

(9)	Most farmers who own a donkey feeds it.

[C [E [[most [farmer [who [1 [[a donkey] [2 [t1 [owns t2]]]]]]]] [3 [it [4 [t3 [feeds t4]]]]]]]]

Result [0]:


((2 * Card_<{e},n>(Set x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))))) > Card_<{e},n>(Set x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e))))))

Result [1]:


((2 * Card_<{e},n>(Set x_e: ((Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))))) > Card_<{e},n>(Set x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e))))))

Result [2]:


((2 * Card_<{e},n>(Set x_e: ((~(Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x_e)) & Own_<(e,e),t>(x_e, x2_e)))) & Farmer_<e,t>(x_e)) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))))) > Card_<{e},n>(Set x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e))))))

Result [3]:


((2 * Card_<{e},n>(Set x_e: ((~(Farmer_<e,t>(x_e) & (Exists x2_e: ((Donkey_<e,t>(x2_e) & ~Feed_<(e,e),t>(x_e, x2_e)) & Own_<(e,e),t>(x_e, x2_e)))) & Farmer_<e,t>(x_e)) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e)))))) > Card_<{e},n>(Set x_e: (Farmer_<e,t>(x_e) & (Exists x2_e: (Donkey_<e,t>(x2_e) & Own_<(e,e),t>(x_e, x2_e))))))


Result [1] is the weak reading we're after, and Result [3] is the strong interpretation.

